# HotelRec Dataset - Exploratory Data Analysis

Full dataset: ~50M TripAdvisor reviews, 365K hotels, 22M users.  
Stats were computed via `scripts/explore_data.py` (streaming, O(1) memory).  
Results saved in `results/data_evaluation.json`.

In [ ]:
import json
import matplotlib.pyplot as plt

with open('../results/data_evaluation.json') as f:
    stats = json.load(f)

basic = stats['basic_stats']
print(f"Total reviews:    {basic['total_reviews']:,}")
print(f"Unique users:     {basic['unique_users']:,}")
print(f"Unique items:     {basic['unique_items']:,}")
print(f"Sparsity:         {basic['sparsity_pct']:.3f}%")
print(f"Avg reviews/user: {basic['avg_reviews_per_user']:.2f}")
print(f"Avg reviews/item: {basic['avg_reviews_per_item']:.2f}")

In [ ]:
# Rating distribution
hist = stats['rating_distribution']['histogram']
stars = [1, 2, 3, 4, 5]
counts = [hist[str(s)] for s in stars]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(stars, counts, color='steelblue')
ax.set_xlabel('Rating')
ax.set_ylabel('Count (millions)')
ax.set_title(f'Rating Distribution (mean={stats["rating_distribution"]["mean"]:.2f})')
ax.set_xticks(stars)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
plt.tight_layout()
plt.show()

In [ ]:
# User activity distribution
buckets = stats['user_activity']['buckets']
labels = list(buckets.keys())
values = list(buckets.values())

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, values, color='coral')
ax.set_xlabel('Reviews per user')
ax.set_ylabel('Number of users')
ax.set_title(f'User Activity ({stats["user_activity"]["users_1_review_pct"]:.1f}% have exactly 1 review)')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# Temporal distribution
temporal = stats['temporal']
years = sorted(temporal.keys())
counts = [temporal[y] for y in years]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(years, counts, color='seagreen')
ax.set_xlabel('Year')
ax.set_ylabel('Reviews')
ax.set_title('Reviews per Year (2001-2019)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Sub-rating coverage
sub = stats['sub_ratings']
names = [k for k in sub if sub[k]['count'] > 0]
coverages = [sub[k]['coverage_pct'] for k in names]
avgs = [sub[k]['avg'] for k in names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.barh(names, coverages, color='mediumpurple')
ax1.set_xlabel('Coverage (%)')
ax1.set_title('Sub-Rating Coverage')

ax2.barh(names, avgs, color='goldenrod')
ax2.set_xlabel('Average Rating')
ax2.set_title('Sub-Rating Averages')
ax2.set_xlim(3.5, 5)
plt.tight_layout()
plt.show()

In [ ]:
# Baseline results comparison
with open('../results/baselines/baseline_results_20core.json') as f:
    baselines = json.load(f)

models = list(baselines.keys())
hr10 = [baselines[m]['HR@10'] for m in models]
ndcg10 = [baselines[m]['NDCG@10'] for m in models]

x = range(len(models))
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([i - 0.15 for i in x], hr10, 0.3, label='HR@10', color='steelblue')
ax.bar([i + 0.15 for i in x], ndcg10, 0.3, label='NDCG@10', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel('Score')
ax.set_title('Baseline Comparison (20-core)')
ax.legend()
ax.set_ylim(0, 0.8)
plt.tight_layout()
plt.show()